# Legacy mode

Legacy mode reproduces the original WFDB package, including the signal
parameters it assumes when a record does not give them. For example, it
reads a record whose header gives no sampling frequency at 250 Hz. Use it
only to reproduce old results. You turn it on in three ways:

* tools: `--legacy` as the first argument;
* high-level functions: `legacy=True`;
* the whole library: `wfdbx.wfdbx_set_legacy(1)`.

[DIVERGENCES.md](https://github.com/alphanumericslab/wfdb-x/blob/main/docs/DIVERGENCES.md#legacy-mode) describes what changes in legacy mode.

In [ ]:
import numpy as np
import wfdbx

# Records are written to, and read from, the current directory (no input
# or output directory is set).

# This example writes a record, nofreq, whose header does not give its
# sampling frequency.
with open("nofreq.hea", "w") as f:
    f.write("nofreq 1\nnofreq.dat 16 200 16 0 0 0 0 sig\n")
with open("nofreq.dat", "wb") as f:
    f.write(np.arange(1000, dtype="<i2").tobytes())

Without legacy mode, a record whose header gives no sampling frequency
has `fs = 0`, and a time in seconds is an error:

In [ ]:
rec = wfdbx.read_record("nofreq")
print("sampling frequency:", rec.fs, "(not specified)")
try:
    wfdbx.read_record("nofreq", sampto="0:1")       # a time needs a frequency
except ValueError as err:
    print("ValueError:", err)

With `legacy=True`, the record is read at 250 Hz:

In [ ]:
old = wfdbx.read_record("nofreq", sampto="0:1", legacy=True)
print("legacy mode assumes", old.fs, "Hz:", len(old.signals), "samples in '1 second'")

The tools behave in the same way. `wrsamp` without `-F`, `-G`, `-R` and
`-U` stops with an error. `wrsamp --legacy` uses the original package's
values. This example writes a record, `legacy1`, in the current directory:

In [ ]:
import subprocess
import sys

text = "".join(f"{i}\n" for i in range(10))
for args in (["-o", "legacy1"], ["--legacy", "-o", "legacy1"]):
    r = subprocess.run([sys.executable, "-m", "wfdbx.cli", "wrsamp", *args],
                       input=text, capture_output=True, text=True)
    print("wrsamp", *args, "->", r.returncode, r.stderr.strip())
print(open("legacy1.hea").read())